In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
file_path = r'/content/drive/MyDrive/Optimal Fleet Sizing/data/'

In [ ]:
!pip install gurobipy

import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from math import radians, sin, cos, sqrt, atan2

In [ ]:
def load_UAM_network(filepath_vpt_loc):
    """
    Load vertiport locations and generate network nodes and indices.

    Parameters:
    - filepath_vpt_loc: Path to vpt_locations.csv containing a 'Location' column.

    Returns:
    - locations: List of vertiport names.
    - flight_corridors: List of (i,j) tuples for all origin-destination pairs.
    - nodes: Combined list of locations and corridors.
    - node_idx: Mapping from node to global index.
    - loc_index: Mapping from vertiport location to its index.
    - n_vpt: Number of vertiports
    """
    df = pd.read_csv(filepath_vpt_loc)
    locations = df['Location'].tolist()
    flight_corridors = [(i, j) for i in locations for j in locations if i != j]
    nodes = locations + flight_corridors
    node_idx = {node: idx for idx, node in enumerate(nodes)}
    loc_index = {loc: idx for idx, loc in enumerate(locations)}
    n_tot = len(nodes)
    n_vpt = len(locations)
    return locations, flight_corri]dors, nodes, node_idx, loc_index, n_tot, n_vpt


def load_input_data(filepath_routing, filepath_time, filepath_dist, filepath_arrival, filepath_relative_throughput):
    """
    Load matrices and vectors:

    Parameters:
    - filepath_routing: Path to routing_matrix.csv
    - filepath_time: Path to travel_time_matrix.csv
    - filepath_dist: Path to distance_matrix.csv
    - filepath_arrival: Path to arrival_rate_vector.csv
    - filepath_relative_throughput: Path to relative_throughput_vector.csv

    Returns:
    - R: routing matrix
    - T: travel-time matrix
    - D: distance matrix
    - lambda_vec: arrival-rate vector
    - pi_vec: relative-throughput vector
    """
    R = np.loadtxt(filepath_routing, delimiter=',')  # shape (|N|,|N|)
    T = np.loadtxt(filepath_time, delimiter=',')     # shape (|V|,|V|)
    D = np.loadtxt(filepath_dist, delimiter=',')     # shape (|V|,|V|)
    lambda_vec = np.loadtxt(filepath_arrival, delimiter=',')  # shape (|V|,)
    pi_vec = np.loadtxt(filepath_relative_throughput, delimiter=',')  # shape (|N|,)
    return R, T, D, lambda_vec, pi_vec

def compute_cost_terms(D: np.ndarray, loc_index: dict, flight_corridors: list, c_fare: float, c_usage: float) -> tuple:
    """
    Compute cost vectors:
    - c_fare_vec: ticket fare per corridor (distance * c_fare)
    - c_usage_vec: vertiport usage fee per operation

    Parameters:
    - D: distance matrix (n_vpt × n_vpt)
    - loc_index: mapping from location to index in D
    - flight_corridors: list of (i,j) tuples for corridors
    - c_fare: fare rate in currency/km
    - c_usage: usage fee per operation

    Returns:
    - c_fare_vec (np.ndarray, size |F|)
    - c_usage_vec (np.ndarray, size |V|)
    """
    # Fare per flight corridor
    c_fare_vec = np.array([D[loc_index[i], loc_index[j]] * c_fare for i, j in flight_corridors]) # shape (|F|,)

    # Usage fee at each vertiport
    n_vpt = len(loc_index)
    c_usage_vec = np.ones(n_vpt) * c_usage # shape (|V|,)
    return c_fare_vec, c_usage_vec

### Closed Jackson Network Modeling

- `compute_traffic_balance_equation`: In the closed Jackson network, the relative throughput $\pi_i$ is satisfied:

$$
  \pi_i = \sum_{j \in N}~\pi_jr_{ij} \quad \forall i \in N
$$
- `compute_routing_matrix`:
Vehicles in the closed Jaskon network move according to the routing matrix $R = [r_{ij}]$ is given by

$$
r_{ij} =
  \begin{cases}
    p_{il} & \quad i \in V, j \in F, i= \text{Entry}(j),~l=\text{Exit}(j) \\[4pt]
    1 & \quad i \in F, j \in V j=\text{Exit}(i) \\[4pt]
    0 & \quad \text{otherwise}
  \end{cases}
$$

In [ ]:
def compute_traffic_balance_equation(
    R: np.ndarray
  ) -> np.ndarray:
    """
    Compute traffic balance equation
    """
    # Compute left eigenvector of R: Solve \pi R = \pi
    evals, evecs = np.linalg.eig(R.T)

    # Find the eigenvalue closest to 1
    idx = np.argmin(np.abs(evals - 1))

    # Extract corresponding eigenvector
    pi_vec = np.real(evecs[:, idx])
    return pi_vec

def compute_routing_matrix(
    nodes: list,
    node_idx: list,
    P: np.ndarry
  ) -> np.ndarry:
    """
    Compute routing matrix bewteen nodes

    Parameter:
    - nodes
    - node_idx
    - P: Choice probability matrix (n_vpt × n_vpt)

    Returns
    - R (np.ndarray, size (|N|,|N|))
    """
    # Initialize routing matrix
    R = np.zeros((len(nodes), len(nodes)))

    # Routing matrix
    for i_node in nodes:
        for j_node in nodes:
            i_idx = node_idx[i_node]
            j_idx = node_idx[j_node]

            # Case : vertiport -> flight corridor
            if isinstance(i_node, str) and isinstance(j_node, tuple):
                entry, exit = j_node
                if i_node == entry:
                    R[i_idx, j_idx] = P[node_idx[entry], node_idx[exit]]

            # Case : flight corridor -> vertiport
            elif isinstance(i_node, tuple) and isinstance(j_node, str):
                entry, exit = i_node
                if j_node == exit:
                    R[i_idx, j_idx] = 1
    return R

def scale_arrival_rate(
    lambda_vec: np.darray,
    lambda_total: float
  ) -> np.darray:
    """
    Scale the arrival rate according to the total demand

    Parameters:
    - lambda_vec
    - lambda_total: arrival rate of paasengers who use UAM modes in the system

    Returns:
    - lambda_vec (np.ndarray, size |V|)
    """
    lambda_props = lambda_vec / np.sum(lambda_vec) # shape (|V|,)
    lambda_vec = lambda_props * lambda_total
    return lambda_vec

def compute_service_rate(
    lambda_vec: np.darray,
    T: np.darray,
    flight_corridors: list,
    loc_index: dict,
    n_tot: float,
    n_vpt: float
  ) -> np.ndarry:
    """
    Compute the service rate for nodes

    Parameters:
    - lambda_vec: np.darray
    - T: np.darray
    - flight_corridors
    - loc_index
    - n_tot
    - n_vpt

    Returns:
    - mu1 (np.ndarray, size |N|)
    """
    mu1 = np.zeros(n_tot) # shape (|N|,)

    # Service rates at vertiports
    mu1[:n_vpt] = lambda_vec

    # Service rates at flight corridors
    base_mu = np.zeros(n_tot)
    for f_idx, (j, k) in enumerate(flight_corridors, start=n_vpt):
        base_mu[f_idx] = 1.0 / T[loc_index[j], loc_index[k]]
    mu1[n_vpt:] = base_mu[vpt:]

    return mu1


### Optimal Rebalancing LP (Zhang & Pavone, 2016)

The optimal rebalancing problem is formulated as the non-linear optimizaiton problem.
- **Objective** : Minimize the  number of rebalancing vehicles in flight corridor:
  $$
    \min_{\psi_i, \alpha_{ij}}~\sum_{i,j}T_{ij}\alpha_{ij},\psi_i
  $$


- **Constraints** : Maintain the relative utilization at all nodes:  

  $$ \gamma_i = \gamma_j \\
\sum_j~\alpha_{ij}=1 \\
\alpha_{ij} \ge 0, \psi \ge 0, \quad \forall i,j \in N
  $$

- **Decision variables** : $\psi_i, \alpha_{ij}$

Pavone showed how to reduce the dimension of the optimization problem and how to convert into the minimum cost flow problem
- **Objective** :
  $$
    \min_{\beta_{ij}}~\sum_{i,j}T_{ij}\,\beta_{ij}
  $$

- **Constraints**  
$$
\sum_{i \neq j}~(\beta_{ij} - \beta_{ji}) = - \lambda_i + \sum_{i \neq j}~p_{ji}\lambda_j \quad
$$

- **Decision variables**  
$$
\beta_{ij} \;\ge\; 0 \quad i,j \in V,~~i \neq j
$$

- where  
  - $T_{ij}$ is the travel time from $i$ to $j$  
  - $\lambda_i$ is the passenger arrival rate at vertiport $i$,  
  - $p_{ij}$ is the choice probability a passenger at $i$ wants to go to $j$.  


In [ ]:
def solve_rebalancing_lp(T: np.ndarray, lambda_vec: np.ndarray, P: np.ndarray, n_vpt: int) -> tuple:
    """
    Solve the optimal rebalancing problem (Zhang and Pavone, 2016) using Gurobi optimizer.

    Decision Variables:
    - beta[i,j] >= 0: number of vehicles rebalanced from i to j per unit time

    Objective:
    - minimize sum_{i!=j} T[i,j] * beta[i,j]

    Constraints:
    - flow conservation at each vertiport i:
        outflow_i - inflow_i = -lambda_vec[i] + sum_j P[j,i] * lambda_vec[j]

    Returns:
    - psi: total outbound rebalancing flow per node (size n_vpt,)
    - alpha: routing proportions matrix (n_vpt x n_vpt)
    - beta_mat: full rebalancing flow matrix (n_vpt x n_vpt)
    """

    m = gp.Model('Rebalancing')
    m.Params.OutputFlag = 0

    # Decision variables beta[i,j]
    beta_vars = {}
    for i in range(n_vpt):
        for j in range(n_vpt):
            if i != j:
                beta_vars[i, j] = m.addVar(lb=0.0, name=f'beta_{i}_{j}')

    # Objective: minimize total travel time cost
    m.setObjective(gp.quicksum(T[i, j] * beta_vars[i, j] for i,j in beta_vars), GRB.MINIMIZE)

    # Flow conservation constraints
    for i in range(n_vpt):
        outflow = gp.quicksum(beta_vars[i, j] for j in range(n_vpt) if j != i)
        inflow = gp.quicksum(beta_vars[j, i] for j in range(n_vpt) if j != i)
        rhs = -lambda_vec[i] + float(np.dot(P[:, i], lambda_vec))
        m.addConstr(outflow - inflow == rhs, name=f'flow_{i}')

    m.optimize()

    # Extract results
    beta_mat = np.zeros((n_vpt, n_vpt))
    for (i, j), var in beta_vars.items():
        beta_mat[i, j] = var.X

    psi = beta_mat.sum(axis=1)
    alpha = np.zeros_like(beta_mat)
    for i in range(n_vpt):
        if psi[i] > 0:
            alpha[i] = beta_mat[i] / psi[i]
    return psi, alpha, beta_mat

**Composite arrival rate matrix**
- Let  $\tilde\lambda_i = \lambda_i + \psi_i$ be the composite arrival rate at vertiport $i$.
- Compute the composite arrival rate matrix $\tilde P = [\tilde p_{ij}]$ by
$$
\tilde p_{ij} =
  \begin{cases}
    \dfrac{\lambda_i\,P_{ij} \;+\; \psi_i\,\alpha_{ij}}{\tilde\lambda_i},
    & \tilde\lambda_i > 0,\; i \neq j,\\[8pt]
    0, & i = j,
  \end{cases}
$$
according to $\sum_j \tilde p_{ij} = 1\ $.  

In [ ]:
def compute_composite_arrival_routing(
    lambda_vec: np.ndarray,
    psi: np.ndarray,
    P: np.ndarray,
    alpha: np.ndarray,
    n_vpt: int,
) -> tuple:
    """
    Compute composite arrival rates and mixed routing probabilities.

    Returns:
    - lambda_tilde: composite arrival vector (lambda_vec and psi)
    - P_tilde: mixed routing probability matrix
    """
    # Compute composite arrival rates
    lambda_tilde = lambda_vec + psi # shape (|V|,)

    # Error if any arrival rate is non-positive
    if np.any(lambda_tilde <= 0):
        zero_idx = np.where(lambda_tilde <= 0)[0]
        raise ValueError(f"Composite arrival rate zero or negative at indices {zero_idx}")

    # Compute composite choice probabilities
    P_tilde = np.zeros((n_vpt, n_vpt)) # shape (|V|, |V|)
    for i in range(n_vpt):
        for j in range(n_vpt):
            if i != j:
                P_tilde[i, j] = (lambda_vec[i] * P[i, j] + psi[i] * alpha[i, j]) / lambda_tilde[i]
        P_tilde[i, i] = 0.0
    return lambda_tilde, P_tilde

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB

# Load vertiport data and define nodes ---
df = pd.read_csv(r'data/vpt_locations.csv')

# Load data
R = np.loadtxt(r'data/routing_matrix.csv', delimiter=',') # shape (|N|,|N|)
T = np.loadtxt(r'data/travel_time_matrix.csv', delimiter=',') # shape (|V|,|V|)
D = np.loadtxt(r'data/distance_matrix.csv', delimiter=',') # shpae (|V|,|V|)

lambda_vec = np.loadtxt(r'data/arrival_rate_vector.csv', delimiter=',') # shape (|V|,)
# c_usage_vec = np.loadtxt(r'data/c_usage_vector.csv', delimiter=',') # shape (|V|,)
c_usage_vec = np.ones(len(locations))
pi_vec = np.loadtxt(r'data/relative_throughput_vector.csv', delimiter=',') # shape (|N|,)

# Input Parameters
tol = 0.5 # minimum vehicle availability
c_fare = 2000 # ticket fare in KRW/km
c_penalty = 10000 # penalty cost in KRW
c_usage = 5000 # vertiport usage fee in KRW/ops
c_mnt = 2500 # maintenance cost in KRW/veh
lambda_total = 200 # arrival rate of paasengers who use UAM modes in the system

# Compute c_fare per corridor: distance * 2000 won/km
c_fare_vec = np.array([D[loc_index[i], loc_index[j]] * c_fare for i, j in flight_corridors]) # shape  (|F|,)
c_usage_vec *= c_usage
c_tilde_vec = c_penalty - c_usage_vec
# print(c_tilde_vec)

# Scale arrival rate
lambda_props = lambda_vec / np.sum(lambda_vec)
lambda_vec = lambda_props * lambda_total
# print(lambda_vec)

# Pre-compute base service rates for corridors
n_vpt = len(locations)
n_total = len(nodes)
base_mu = np.zeros(n_total)
for f_idx, (j, k) in enumerate(flight_corridors, start=n_vpt):
    base_mu[f_idx] = 1.0 / T[loc_index[j], loc_index[k]]
# print(base_mu)

# Service Rate
# Compute mu_i(1) for all nodes
mu_1 = np.zeros(n_total)
mu_1[:n_vpt] = lambda_vec # for vertiports: mu_i(1) = lambda_i
mu_1[n_vpt:] = base_mu[n_vpt:] # for corridors: mu_i(1) = 1 * base_mu_i
# print(mu_n1)

# Compute gamma_base and gamma_tilde for corridors
pi_vpt = pi_vec[:n_vpt] # relative_throughput at vertiports
pi_cdr = pi_vec[n_vpt:] # relative throughput at flight corridors

gamma_base = pi_cdr / mu_1[n_vpt:]

# gamma_tilde_i = gamma_base_i + (sum_{v in V} c_tilde_v * pi_v) / (sum_{f in F} c_fare_f)
numerator = np.dot(c_tilde_vec, pi_vpt)
denominator = np.sum(c_fare)
gamma_vec = gamma_base + numerator / denominator
# print(gamma_vec)

# --- 2) Gurobi model for rebalancing LP ---
model = gp.Model('rebalancing')
model.Params.OutputFlag = 0  # silent

beta = {}
for i in range(n_vpt):
    for j in range(n_vpt):
        if i != j:
            beta[i, j] = model.addVar(lb=0.0, name=f'beta_{i}_{j}')

model.setObjective(
    gp.quicksum(T[i, j] * beta[i, j] for (i, j) in beta),
    GRB.MINIMIZE
)

for i in range(n_vpt):
    outflow = gp.quicksum(beta[i, j] for j in range(n_vpt) if j != i)
    inflow  = gp.quicksum(beta[j, i] for j in range(n_vpt) if j != i)
    rhs = -lambda_vec[i] + np.dot(P[:, i], lambda_vec)
    model.addConstr(outflow - inflow == rhs)

model.optimize()

# Extract beta_mat, psi, alpha
beta_mat = np.zeros((n_vpt, n_vpt))
for (i, j), var in beta.items():
    beta_mat[i, j] = var.X
psi = beta_mat.sum(axis=1)
alpha = np.zeros_like(beta_mat)
for i in range(n_vpt):
    if psi[i] > 0:
        alpha[i] = beta_mat[i] / psi[i]

# --- 3) Composite arrival rates and routing ---
lambda_tilde = lambda_vec + psi
p_tilde = np.zeros((n_vpt, n_vpt))
for i in range(n_vpt):
    total = lambda_tilde[i]
    if total > 0:
        # compute mixed probabilities for i → j
        for j in range(n_vpt):
            if i != j:
                p_tilde[i, j] = (lambda_vec[i] * P[i, j] + psi[i] * alpha[i, j]) / total
        p_tilde[i, i] = 0.0
        # renormalize row i
        row_sum = p_tilde[i].sum()
        if row_sum > 0:
            p_tilde[i] /= row_sum
        else:
            # fallback to uniform if all zero
            for j in range(n_vpt):
                if i != j:
                    p_tilde[i, j] = 1.0/(n_vpt-1)
            p_tilde[i, i] = 0.0
    else:
        # if no arrivals, copy original probabilities
        p_tilde[i] = P[i].copy()
        p_tilde[i, i] = 0.0
        p_tilde[i] /= p_tilde[i].sum()

# 4) Recompute SS stationary distribution ---
eigvals, eigvecs = np.linalg.eig(p_tilde.T)
pi_ss = np.real(eigvecs[:, np.isclose(eigvals, 1.0)].flatten())
pi_ss /= pi_ss.sum()

# 5) Build full pi_vec (including corridors) ---
pi_vec_new = np.zeros(n_total)
pi_vec_new[:n_vpt] = pi_ss
for k, (i, j) in enumerate(flight_corridors):
    idx = n_vpt + k
    pi_vec_new[idx] = pi_ss[loc_index[i]] * p_tilde[loc_index[i], loc_index[j]]

# Update service rates and routing for ESS
mu_1[:n_vpt] = lambda_tilde
pi_vec[:] = pi_vec_new

# --- 6) Exact Solution Search (ESS) ---
obj_history, n_history = [], []
availability_history, avg_availability_history = [], []
W = [np.zeros(n_total)]
L = [np.zeros(n_total)]
n = 0
m_min = -1
obj_prev = 0

while n <= 100:
    n += 1
    # MVA: compute response times W_vector
    W_vector = np.zeros(n_total)
    for node in nodes:
        idx = node_idx[node]
        if isinstance(node, tuple):
            W_vector[idx] = 1.0 / mu_1[idx]
        else:
            W_vector[idx] = (1.0 + L[n-1][idx]) / lambda_vec[loc_index[node]]
    Xn = n / np.dot(pi_vec, W_vector)
    L_vector = Xn * pi_vec * W_vector

    # availability at each vertiport
    A = np.array([Xn * pi_vec[node_idx[loc]] / lambda_vec[loc_index[loc]]
                  for loc in locations])
    avg_availability_history.append(A.mean())
    W.append(W_vector)
    L.append(L_vector)

    # objective: fare revenue - usage cost - maintenance cost
    L_corridors = L_vector[n_vpt:]
    Lambda_vertiports = (Xn * pi_vec)[:n_vpt]
    obj_n = np.dot(c_fare_vec, L_corridors) \
            - np.dot(c_usage_vec, Lambda_vertiports) \
            - c_mnt * n

    n_history.append(n)
    obj_history.append(obj_n)

    if m_min == -1 and all(a >= tol for a in A):
        m_min = n

    if m_min != -1 and obj_n < obj_prev:
        m_star = n - 1
        break
    obj_prev = obj_n

    # Step 3: objective evaluation
    if m_min != -1 and n >= m_min:
        # new objective:
        # sum_{F} c_fare_i * L_i(n)
        # - sum_{V} c_usage_i * Lambda_i(n)
        # - c_mnt * n
        L_corridors = L_vector[n_vpt:]
        Lambda_vector = Xn * pi_vec
        Lambda_vertiports = Lambda_vector[:n_vpt]

        obj_n = np.dot(c_fare_vec, L_corridors) \
                - np.dot(c_usage_vec, Lambda_vertiports) \
                - c_mnt * n

        n_history.append(n)
        obj_history.append(obj_n)

        if obj_n < obj_prev:
            m_star = n - 1
            break
        obj_prev = obj_n


print(f"Minimum fleet size m_min = {m_min}")
print(f"Optimal fleet size m* = {m_star}")
print(f"Objective value = {obj_prev}")

# --- 7) Plot avg queue length and response time at vertiports ---
avg_L = [L[m][:n_vpt].mean() for m in range(1, m_star+1)]
avg_W = [(W[m][:n_vpt] * 60).mean() for m in range(1, m_star+1)]

plt.figure()
plt.plot(range(1, m_star+1), avg_L, marker='o')
plt.xlabel('Fleet size (m)')
plt.ylabel('Average queue length at vertiports')
plt.grid(True)

plt.figure()
plt.plot(range(1, m_star+1), avg_W, marker='o')
plt.xlabel('Fleet size (m)')
plt.ylabel('Average response time at vertiports (minutes)')
plt.grid(True)

plt.figure()
plt.plot(n_history, obj_history)
plt.xlabel('Fleet size (m)')
plt.ylabel('Objective function obj(m)')
# plt.title('Objective vs Fleet Size')
plt.tight_layout()
plt.show()

plt.show()


In [ ]:
final_availability = A[-1]
plt.figure()
plt.bar(locations, final_availability)
plt.xlabel('Vertiport')
plt.ylabel('Availability')
# plt.title(f'Service Availability at Optimal Fleet Size (n={m_star})')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def exact_solution_search_algorithm(
    c_penalty=0,
    vpt_csv='data/vpt_locations.csv',
    routing_csv='data/routing_matrix.csv',
    time_csv='data/travel_time_matrix.csv',
    dist_csv='data/distance_matrix.csv',
    arrival_csv='data/arrival_rate_vector.csv',
    throughput_csv='data/relative_throughput_vector.csv',
    tol=0.5,
    c_fare=2000,
    c_usage=5000,
    c_mnt=2500,
    lambda_total=200,
    max_m=100
):

    # Load data and set up nodes/corridors
    df        = pd.read_csv(vpt_csv)
    locs      = df['Location'].tolist()
    corridors = [(i, j) for i in locs for j in locs if i != j]
    n_vpt     = len(locs)
    n_nodes   = n_vpt + len(corridors)
    loc_idx   = {loc: i for i, loc in enumerate(locs)}
    node_list = locs + corridors
    node_idx  = {node: i for i, node in enumerate(node_list)}

    # Load matrices and vectors
    T          = np.loadtxt(time_csv,    delimiter=',')
    D          = np.loadtxt(dist_csv,    delimiter=',')
    lambda_raw = np.loadtxt(arrival_csv, delimiter=',')
    pi_vec     = np.loadtxt(throughput_csv, delimiter=',')

    # Cost and fare vectors
    c_usage_vec = np.ones(n_vpt) * c_usage
    fare_vec    = np.array([D[loc_idx[i], loc_idx[j]] * c_fare
                             for i, j in corridors])

    # Scale arrival rates to total
    props      = lambda_raw / lambda_raw.sum()
    lambda_vec = props * lambda_total

    # Compute mu_1 for each node
    base_mu = np.zeros(n_nodes)
    for idx, (i, j) in enumerate(corridors, start=n_vpt):
        base_mu[idx] = 1.0 / T[loc_idx[i], loc_idx[j]]
    mu_1 = np.zeros(n_nodes)
    mu_1[:n_vpt]   = lambda_vec     # vertiports
    mu_1[n_vpt:]   = base_mu[n_vpt:]  # corridors

    # Perform MVA-ESS to build history
    n_history = []
    obj_history = []
    L_prev = np.zeros(n_nodes)
    obj_prev = -np.inf

    for n in range(1, max_m + 1):
        # 1) MVA: compute response times W and queue lengths L
        W = np.zeros(n_nodes)
        for node in node_list:
            idx = node_idx[node]
            if isinstance(node, tuple):
                W[idx] = 1.0 / mu_1[idx]
            else:
                W[idx] = (1 + L_prev[idx]) / lambda_vec[loc_idx[node]]
        Xn = n / np.dot(pi_vec, W)
        L  = Xn * pi_vec * W

        # 2) Compute penalty term from lost arrivals
        A = np.array([Xn * pi_vec[node_idx[loc]] / lambda_vec[loc_idx[loc]] for loc in locs])
        lost_rate = lambda_vec * (1 - A)
        penalty_term = c_penalty * lost_rate.sum()

        # 3) Objective function including penalty
        Lambda_v = Xn * pi_vec[:n_vpt]
        obj = (
            fare_vec.dot(L[n_vpt:])    # fare revenue
          - c_usage_vec.dot(Lambda_v)  # usage fee
          - c_mnt * n                  # maintenance cost
          - penalty_term               # penalty cost
        )

        # 4) Record history
        n_history.append(n)
        obj_history.append(obj)
        L_prev = L

        # 5) Stop if objective decreases
        if obj < obj_prev:
            m_star = n - 1
            break
        obj_prev = obj

    return n_history, obj_history, m_star


# Plotting
penalties   = [0, 500, 1000]
linestyles  = ['-', '--', '-.']
markers     = ['o', 's', '^']

plt.figure(figsize=(10, 6))
for cp, ls, mk in zip(penalties, linestyles, markers):
    n_hist, obj_hist, m_star = exact_solution_search_algorithm(cp)
    # Print optimal results for each penalty
    print(f"c_penalty={cp}: optimal fleet size m*={m_star}, obj*={obj_hist[m_star-1]:.2f}")

    # Plot the objective curve
    plt.plot(
        n_hist, obj_hist,
        linestyle=ls,
        marker=mk,
        markersize=10,
        label=fr'$c_{{pen}}={cp}$'  # LaTeX-formatted legend
    )

# Configure axes and grid
plt.xlabel('Fleet size (m)', fontsize=14)
plt.ylabel('Objective function $obj(m)$', fontsize=14)
# plt.title('Objective vs Fleet Size with Penalty Term\nand Optimal Annotations', fontsize=16)
plt.grid(True, linewidth=1.2)
plt.legend(fontsize=14)
plt.tight_layout()
plt.savefig('result/fig_penalty_scheme')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def compute_obj_history(
    c_penalty,
    vpt_csv='data/vpt_locations.csv',
    routing_csv='data/routing_matrix.csv',
    time_csv='data/travel_time_matrix.csv',
    dist_csv='data/distance_matrix.csv',
    arrival_csv='data/arrival_rate_vector.csv',
    throughput_csv='data/relative_throughput_vector.csv',
    tol=0.5,
    c_fare=2000,
    c_usage=5000,
    c_mnt=2500,
    lambda_total=200,
    max_m=100
):
    # --- Load data and set up ---
    df        = pd.read_csv(vpt_csv)
    locs      = df['Location'].tolist()
    corridors = [(i,j) for i in locs for j in locs if i!=j]
    n_vpt     = len(locs)
    n_nodes   = n_vpt + len(corridors)
    loc_idx   = {loc:i for i,loc in enumerate(locs)}
    node_list = locs + corridors
    node_idx  = {node:i for i,node in enumerate(node_list)}

    T          = np.loadtxt(time_csv,    delimiter=',')
    D          = np.loadtxt(dist_csv,    delimiter=',')
    lambda_raw = np.loadtxt(arrival_csv, delimiter=',')
    pi_vec     = np.loadtxt(throughput_csv, delimiter=',')

    # cost & fare vectors
    c_usage_vec = np.ones(n_vpt) * c_usage
    fare_vec    = np.array([D[loc_idx[i],loc_idx[j]]*c_fare for i,j in corridors])

    # scale arrival rates
    props      = lambda_raw / lambda_raw.sum()
    lambda_vec = props * lambda_total

    # compute mu_1
    base_mu = np.zeros(n_nodes)
    for idx,(i,j) in enumerate(corridors, start=n_vpt):
        base_mu[idx] = 1.0 / T[loc_idx[i], loc_idx[j]]
    mu_1 = np.zeros(n_nodes)
    mu_1[:n_vpt]   = lambda_vec
    mu_1[n_vpt:]   = base_mu[n_vpt:]

    # histories
    n_history    = []
    obj_history  = []
    avgA_history = []
    avgL_history = []
    avgW_history = []

    L_prev   = np.zeros(n_nodes)
    obj_prev = -np.inf

    for n in range(1, max_m+1):
        # 1) MVA response times W & queue lengths L
        W = np.zeros(n_nodes)
        for node in node_list:
            idx = node_idx[node]
            if isinstance(node, tuple):
                W[idx] = 1.0 / mu_1[idx]
            else:
                W[idx] = (1 + L_prev[idx]) / lambda_vec[loc_idx[node]]
        Xn = n / np.dot(pi_vec, W)
        L  = Xn * pi_vec * W

        # 2) average availability
        A = np.array([ Xn*pi_vec[node_idx[loc]]/lambda_vec[loc_idx[loc]]
                       for loc in locs ])
        avgA = A.mean()

        # 3) avg queue length & response time at vertiports
        avgL = L[:n_vpt].mean()
        avgW = (W[:n_vpt]*60).mean()

        # 4) penalty term
        lost_rate   = lambda_vec * (1 - A)
        penalty_term = c_penalty * lost_rate.sum()

        # 5) objective
        Lambda_v = Xn * pi_vec[:n_vpt]
        obj = (
            fare_vec.dot(L[n_vpt:])     # revenue
          - c_usage_vec.dot(Lambda_v)   # usage fee
          - c_mnt * n                   # maintenance
          - penalty_term                # penalty
        )

        # record
        n_history.append(n)
        obj_history.append(obj)
        avgA_history.append(avgA)
        avgL_history.append(avgL)
        avgW_history.append(avgW)

        L_prev = L
        if obj < obj_prev:
            m_star = n - 1
            break
        obj_prev = obj

    return n_history, obj_history, avgA_history, avgL_history, avgW_history, m_star


# ====================== Plotting ======================
# 3 penalty cases
penalties  = [0, 5000, 10000]
styles     = ['-', '--', '-.']
markers    = ['o', 's', '^']

# apply global style
plt.rcParams.update({
    'font.size':       10,
    'font.weight':     'medium',
    'axes.labelweight':'medium',
    'font.family':     'sans-serif',
    'font.sans-serif': ['Arial'],
})

# prepare containers
all_results = {}
for cp in penalties:
    all_results[cp] = compute_obj_history(cp)


# 1) Average vehicle availability
plt.figure()
for cp, ls, mk in zip(penalties, styles, markers):
    n_hist, _, avgA, *_ , _ = all_results[cp]
    plt.plot(n_hist, avgA, linestyle=ls, marker=mk, markersize=8, label=f'c_penalty={cp}')
plt.xlabel('Fleet size (m)', fontsize=16)
plt.ylabel('Average vehicle availability', fontsize=16)
plt.xticks(fontsize=14); plt.yticks(fontsize=14)
plt.grid(True, linewidth=1.2)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig("result/fig_avg_veh_availability.png", dpi=600, bbox_inches='tight')
plt.show()

# 2) Average queue length
plt.figure()
for cp, ls, mk in zip(penalties, styles, markers):
    n_hist, _, *_, avgL, _ = all_results[cp]
    plt.plot(n_hist, avgL, linestyle=ls, marker=mk, markersize=8, label=f'c_penalty={cp}')
plt.xlabel('Fleet size (m)', fontsize=16)
plt.ylabel('Average queue length at vertiports', fontsize=16)
plt.xticks(fontsize=14); plt.yticks(fontsize=14)
plt.grid(True, linewidth=1.2)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig("result/fig_avg_queue_length.png", dpi=600, bbox_inches='tight')
plt.show()

# 3) Average response time
plt.figure()
for cp, ls, mk in zip(penalties, styles, markers):
    n_hist, _, *_, avgW, _ = all_results[cp]
    plt.plot(n_hist, avgW, linestyle=ls, marker=mk, markersize=8, label=f'c_penalty={cp}')
plt.xlabel('Fleet size (m)', fontsize=16)
plt.ylabel('Average response time at vertiports (min)', fontsize=16)
plt.xticks(fontsize=14); plt.yticks(fontsize=14)
plt.grid(True, linewidth=1.2)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig("result/fig_avg_response_time.png", dpi=600, bbox_inches='tight')
plt.show()

# 4) Objective function curve
plt.figure()
for cp, ls, mk in zip(penalties, styles, markers):
    n_hist, obj_hist, *_, m_star = all_results[cp]
    plt.plot(n_hist, obj_hist, linestyle=ls, marker=mk, markersize=8, label=f'c_penalty={cp}')
plt.xlabel('Fleet size (m)', fontsize=16)
plt.ylabel('Objective function obj(m)', fontsize=16)
plt.xticks(fontsize=14); plt.yticks(fontsize=14)
plt.grid(True, linewidth=1.2)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig("result/fig_obj_function.png", dpi=600, bbox_inches='tight')
plt.show()
